# Training and Evaluation of the Logarithmic HIV Viral Load Classification Model

This notebook trains and evaluates a convolutional neural network for classifying HIV viral load images into logarithmically spaced concentration categories. The model uses reduced 7-frame tensor inputs derived from time-series microscopy data. Each input tensor contains temporal information from selected frames of the original image sequence, allowing the network to learn spatial-temporal features associated with viral load intensity.

The workflow includes:

1. Loading reduced `.pt` tensor datasets from training, validation, and testing folders.
2. Defining a custom PyTorch dataset class for 7-frame tensor inputs.
3. Modifying a ResNet architecture to accept 7 input channels instead of standard RGB images.
4. Training the model using class-balanced sampling.
5. Generating visualizations, including a confusion matrix and predicted-class distribution plot.

The logarithmic labeling strategy groups samples into ordered viral load categories spanning the assay dynamic range.

## 1. Import required libraries

This section imports the core Python packages used throughout the notebook. PyTorch is used for dataset handling, model construction, optimization, and inference. Torchvision provides the ResNet backbone. Scikit-learn is used for performance evaluation, while Matplotlib and Seaborn are used to generate summary figures.

In [ ]:
import os
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

## 2. Define the reduced tensor dataset

The input data for this model are stored as PyTorch `.pt` tensor files. Each file corresponds to one reduced image sequence and is assigned to a class based on its parent folder.

The logarithmic classification scheme uses five ordered classes:

- `0.undetectable`
- `1.low`
- `2.medium`
- `3.high`
- `4.very high`

Each tensor is expected to contain 7 temporal channels. These channels represent a reduced version of the original image sequence and preserve selected spatial-temporal information relevant to viral load classification.

The custom `PTDataset` class scans the class folders, associates each file with its class label, loads the tensors, removes an extra singleton channel dimension when present, and returns `(tensor, label)` pairs for use with PyTorch dataloaders.

In [ ]:
class PTDataset(Dataset):
    def __init__(self, root_dir, target_size=(500, 500), transform=None):
        """
        Args:
            root_dir (str): Path to the reduced dataset directory.
            target_size (tuple): Kept for compatibility, but not used because
                                 reduced tensors are already resized.
            transform (callable, optional): Optional transformations.
        """
        self.root_dir = root_dir
        self.target_size = target_size
        self.transform = transform
        self.classes = ['0.undetectable', '1.low', '2.medium', '3.high', '4.very high']

        # Collect all file paths and labels
        self.file_list = []
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            if not os.path.exists(class_path):
                continue

            for file in os.listdir(class_path):
                if file.endswith('.pt'):
                    full_path = os.path.join(class_path, file)
                    class_index = self.classes.index(label)
                    self.file_list.append((full_path, class_index))

        # Pre-load everything into memory
        self.data_list = []
        for file_path, label in self.file_list:
            # Load reduced tensor from disk
            tensor_data = torch.load(file_path, map_location='cpu')

            # Reduced files should already be [1, 7, H, W] or [7, H, W]
            if tensor_data.dim() == 4 and tensor_data.shape[0] == 1:
                tensor_data = tensor_data.squeeze(0)  # [7, H, W]

            # Optional transform
            if self.transform:
                tensor_data = self.transform(tensor_data)

            # Store (tensor, label)
            self.data_list.append((tensor_data, label))

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

## 3. Define the ResNet model and training utilities

This section defines the neural network architecture and the functions used for model training and evaluation.

A ResNet model is used as the classification backbone. Because the input tensors contain 7 temporal channels rather than 3 RGB channels, the first convolutional layer is replaced with a new convolutional layer that accepts 7 input channels. The final fully connected layer is also replaced so that the model outputs one prediction for each viral load class.

The training loop uses cross-entropy loss for multi-class classification. During training, model performance is evaluated on the validation set after each epoch. Automatic mixed precision is enabled when CUDA is available to improve GPU training efficiency.

In [ ]:
def get_resnet_model(num_classes=5, input_channels=7, dropout_rate=0.4955157772921656):
    """
    Build ResNet18 with a custom first conv layer
    that expects `input_channels` and adds a Dropout layer.

    model_depth = 18 (ResNet18)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Replace first conv to match your input_channels
    model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    # Replace FC layer to include Dropout before classification
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),  # Dropout before final classification
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model


def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Use AMP if on GPU
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=25):
    """
    Basic training routine using CrossEntropyLoss
    for single-label, multi-class classification.
    """
    
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total

        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    print("Training complete.")

## 4. Configure dataset paths and training parameters

The following section specifies the locations of the training, validation, and testing datasets. Each dataset folder is expected to contain one subfolder per class.

```python
/workspace/data/logarithmic/Training
/workspace/data/logarithmic/Validation
/workspace/data/logarithmic/Testing

In [ ]:
DATA_ROOT = Path("/home/jovyan/work/data/logarithmic")
OUTPUT_ROOT = Path("/home/jovyan/work/results")
MODEL_ROOT = Path("/home/jovyan/work/models")

MODEL_PATH = MODEL_ROOT / "logarithmic_resnet18.pth"

TRAIN_DIR = DATA_ROOT / "Training"
VAL_DIR = DATA_ROOT / "Validation"
TEST_DIR = DATA_ROOT / "Testing"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)


## 5. Train and evaluate the final model

The main training routine performs the following steps:

1. Detects whether CUDA is available and selects the appropriate device.
2. Loads the training, validation, and testing datasets.
3. Computes class-balanced sampling weights from the training labels.
4. Trains the ResNet model using the selected hyperparameters.
5. Evaluates each trained model on the validation set.
6. Saves the model state with the highest validation accuracy.
7. Evaluates the selected model on the held-out test set.

In [ ]:
def main():
    #Check device for CUDA or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    
    train_dataset = PTDataset(root_dir=TRAIN_DIR, target_size=(500, 500))
    val_dataset   = PTDataset(root_dir=VAL_DIR,   target_size=(500, 500))
    test_dataset  = PTDataset(root_dir=TEST_DIR,  target_size=(500, 500))

   
    train_labels = [label for _, label in train_dataset.data_list]
    print("Labels in dataset:", set(train_labels))

    class_counts = Counter(train_labels)
    weights = [1.0 / class_counts[label] for label in train_labels]
    
    train_sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )

    
    use_pin_memory = (device.type == 'cuda')
    batch_size = 32 
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    
    best_val_acc = -float('inf')
    best_model_state = None

    # Train/Evaluate 5 times
    for run_idx in range(5):
        print(f"\n=== Training Run {run_idx+1} of 5 ===")

        
        model = get_resnet_model(num_classes=5, input_channels=7)  
        model.to(device)

        criterion = nn.CrossEntropyLoss()

        learning_rate = 0.00010693471395379046
        weight_decay  = 2.366898587557215e-06
        gamma_rate    = 0.9401941147532752

        optimizer = optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )

        scheduler = optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=gamma_rate)

        
        num_epochs = 18
        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=num_epochs
        )

        # Evaluate on validation set
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        print(f"Run {run_idx+1} validation accuracy: {val_acc:.2f}%")

        # Keep track of best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            print(f"New best model found with val_acc={val_acc:.2f}% (Run {run_idx+1}).")

    # After all 5 runs, save only the best model
    if best_model_state is not None:
        torch.save(best_model_state, OUTPUT_ROOT / "logarithmic_resnet18_best.pth")
        print(f"\nBest model saved with val_acc={best_val_acc:.2f}%")

        best_model = get_resnet_model(num_classes=5, input_channels=7)
        best_model.load_state_dict(best_model_state)
        best_model.to(device)

        test_loss, test_acc = evaluate_model(best_model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}% for the best model")

if __name__ == "__main__":
    main()


## 6. Generate model evaluation figures

This section reloads a trained model and applies it to the held-out testing dataset. Predictions are compared against the ground-truth labels to generate a combined evaluation figure.

The left panel shows a confusion matrix, which summarizes classification accuracy across the five logarithmic viral load categories.

The right panel shows the relationship between the model-predicted class and the true numeric viral load parsed from each file name. Correct predictions are plotted as colored points, while incorrect predictions are marked with red crosses. The y-axis is shown on a logarithmic scale to reflect the wide dynamic range of viral load values.

Together, these plots show both categorical performance and how classification errors are distributed across the underlying continuous viral load range.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# -----------------------------
# Global plot settings
# -----------------------------
plt.rcParams["svg.fonttype"] = "none"


# -----------------------------
# Model loading
# -----------------------------
def load_resnet18_model(
    model_path,
    num_classes=5,
    input_channels=7,
    dropout_rate=0.243493213909431
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    model.conv1 = torch.nn.Conv2d(
        input_channels,
        64,
        kernel_size=7,
        stride=2,
        padding=3,
        bias=False
    )

    model.fc = torch.nn.Sequential(
        torch.nn.Dropout(p=dropout_rate),
        torch.nn.Linear(model.fc.in_features, num_classes)
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    return model, device



### 6.1 Run inference on the testing dataset

The inference function applies the trained model to each sample in the testing dataset and stores three outputs:

1. The true class label.
2. The predicted class label.
3. The raw numeric viral load parsed from the sample file name.

The raw viral load values are used only for visualization, allowing the categorical model predictions to be displayed against the underlying continuous viral load scale.

In [ ]:
# -----------------------------
# Inference helper
# -----------------------------
def run_inference(model, dataset, loader, device):
    """
    Returns:
        true_classes: class indices from dataset labels
        pred_classes: predicted class indices
        raw_labels: numeric viral load labels parsed from filename
    """
    true_classes = []
    pred_classes = []
    raw_labels = []

    model.eval()

    sample_index = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(inputs)
            preds = outputs.argmax(dim=1)

            batch_size = inputs.size(0)

            true_classes.extend(labels.cpu().numpy())
            pred_classes.extend(preds.cpu().numpy())

            for i in range(batch_size):
                file_path, _ = dataset.file_list[sample_index + i]
                filename = os.path.basename(file_path)

                # Assumes filename begins with raw viral load, e.g. 12345_sample.pt
                raw_label = int(filename.split("_")[0])
                raw_labels.append(raw_label)

            sample_index += batch_size

    return (
        np.array(true_classes),
        np.array(pred_classes),
        np.array(raw_labels)
    )


# -----------------------------
# Combined figure
# -----------------------------
def plot_combined_confusion_and_swarm(
    true_classes,
    pred_classes,
    raw_labels,
    class_names,
    class_tick_labels,
):
    is_correct = pred_classes == true_classes

    df = pd.DataFrame({
        "TrueLabel": raw_labels,
        "PredClass": [class_names[p] for p in pred_classes],
        "PredClassIndex": pred_classes,
        "Correct": is_correct
    })

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(10, 4),
        constrained_layout=True
    )

    # -----------------------------
    # Left panel: confusion matrix
    # -----------------------------
    conf_matrix = confusion_matrix(true_classes, pred_classes)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=conf_matrix,
        display_labels=class_tick_labels
    )

    disp.plot(
        cmap=plt.cm.Blues,
        ax=axes[0],
        colorbar=False
    )

    axes[0].set_xlabel("Predicted HIV Viral Load", fontsize=11)
    axes[0].set_ylabel("True HIV Viral Load", fontsize=11)
    axes[0].set_xticklabels(class_tick_labels, fontname="Arial", rotation=45, ha="right")
    axes[0].set_yticklabels(class_tick_labels, fontname="Arial")

    # -----------------------------
    # Right panel: predicted class vs true viral load
    # -----------------------------
    sns.swarmplot(
        data=df[df["Correct"]],
        x="PredClass",
        y="TrueLabel",
        hue="PredClassIndex",
        palette="Set2",
        size=6,
        ax=axes[1],
        legend=False
    )

    sns.swarmplot(
        data=df[~df["Correct"]],
        x="PredClass",
        y="TrueLabel",
        color="red",
        marker="x",
        size=8,
        linewidth=1.5,
        ax=axes[1]
    )

    axes[1].set_yscale("log")
    axes[1].set_ylim(10, 2_010_000)

    axes[1].set_xlabel("Predicted HIV Viral Load", fontsize=11)
    axes[1].set_ylabel("True HIV Viral Load", fontsize=11)

    axes[1].set_xticklabels(class_tick_labels, fontname="Arial", rotation=45, ha="right")
    axes[1].set_yticks([10, 100, 1000, 10000, 100000])
    axes[1].set_yticklabels(["10¹", "10²", "10³", "10⁴", "10⁵"], fontname="Arial")

    axes[1].grid(False)

    fig.savefig(OUTPUT_ROOT / "logarithmic_confusion_swarm.png", dpi=300, bbox_inches="tight")
    plt.show()




### 6.2 Apply the trained model to the test set

The trained model is loaded from disk and evaluated on the final testing dataset. The resulting predictions are used to generate the combined confusion matrix and prediction distribution figure.

In [ ]:
# -----------------------------
# Run
# -----------------------------
model_path = OUTPUT_ROOT / "logarithmic_resnet18_best.pth"
test_dataset_path = TEST_DIR

class_names = [
    "0.undetectable",
    "1.low",
    "2.medium",
    "3.high",
    "4.very high"
]

class_tick_labels = [
    "<10²",
    "10²-10³",
    "10³-10⁴",
    "10⁴-10⁵",
    ">10⁵"
]

model, device = load_resnet18_model(model_path)

test_dataset = PTDataset(
    root_dir=test_dataset_path,
    target_size=(500, 500)
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

true_classes, pred_classes, raw_labels = run_inference(
    model=model,
    dataset=test_dataset,
    loader=test_loader,
    device=device
)

plot_combined_confusion_and_swarm(
    true_classes=true_classes,
    pred_classes=pred_classes,
    raw_labels=raw_labels,
    class_names=class_names,
    class_tick_labels=class_tick_labels,
)